# Meeting Extraction Evaluation: Base vs Fine-tuned Model

**Best Checkpoint:** checkpoint-31500 (validation loss: 0.225720)

This notebook evaluates  fine-tuned Qwen 2.5 14B model against the base model.

## Setup

In [ ]:
!pip install modelscope

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 132.9 MB/s eta 0:00:00


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q pandas tqdm

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.5/376.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.3/181.3 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 123.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 2.7 MB/s et

In [ ]:
import os
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
os.environ['UNSLOTH_SKIP_STATISTICS'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '600'

In [ ]:
import json
import torch
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
from unsloth import FastLanguageModel
from typing import Dict, List, Tuple

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Configuration

In [ ]:
# Paths
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN")  # Set your Hugging Face token as an environment variable

DATA_DIR = "/content/drive/MyDrive/meeting-data"
CHECKPOINT_DIR = "/content/drive/MyDrive/qwen-meeting-extraction"

# Best checkpoint based on training curve
CHECKPOINT_NAME = "checkpoint-3150"  # Lowest val loss: 0.225720

TEST_FILE = f"{DATA_DIR}/test.jsonl"
CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/{CHECKPOINT_NAME}"

print(f"Test file: {TEST_FILE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")

Test file: /content/drive/MyDrive/meeting-data/test.jsonl
Checkpoint: /content/drive/MyDrive/qwen-meeting-extraction/checkpoint-3150


## Load Test Data

In [ ]:
# Load test data
test_data = []
with open(TEST_FILE, 'r') as f:
    for line in f:
        test_data.append(json.loads(line))

print(f"✓ Loaded {len(test_data)} test examples")

# Optional: Use subset for quick testing (comment out for full evaluation)
# test_data = test_data[:20]
# print(f"  Using first {len(test_data)} examples for quick testing")

✓ Loaded 709 test examples


## Load Models

In [ ]:
print("="*80)
print("LOADING BASE MODEL")
print("="*80)

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
)
FastLanguageModel.for_inference(base_model)

print("✓ Base model loaded\n")

LOADING BASE MODEL
==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✓ Base model loaded



In [ ]:
print("="*80)
print("LOADING FINE-TUNED MODEL")
print("="*80)
print(f"Checkpoint: {CHECKPOINT_PATH}\n")

from peft import PeftModel

# Step 1: Load base model
print("Loading base model...")
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,  # Qwen/Qwen2.5-14B-Instruct
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
)

# Step 2: Load LoRA adapters from checkpoint
print("Loading LoRA adapters from checkpoint...")
ft_model = PeftModel.from_pretrained(
    ft_model,
    CHECKPOINT_PATH,
)

# Step 3: Set to inference mode
FastLanguageModel.for_inference(ft_model)

print(" Fine-tuned model loaded (base + trained LoRA adapters)")

LOADING FINE-TUNED MODEL
Checkpoint: /content/drive/MyDrive/qwen-meeting-extraction/checkpoint-3150

Loading base model...
==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading LoRA adapters from checkpoint...
 Fine-tuned model loaded (base + trained LoRA adapters)


## Run Evaluation

In [ ]:
# CELL 1: Helper Functions
# ============================================================================
import json
import re
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple

def parse_json(text: str) -> Tuple[Dict, bool]:
    """Extract and parse JSON from model output"""
    text = text.strip()

    # Remove markdown code blocks
    if "```json" in text:
        text = text.split("```json")[1].split("```")[0].strip()
    elif "```" in text:
        text = text.split("```")[1].split("```")[0].strip()

    # Extract JSON object
    json_match = re.search(r'\{.*\}', text, re.DOTALL)
    if json_match:
        text = json_match.group(0)

    try:
        parsed = json.loads(text)
        return parsed, True
    except:
        return {}, False


def normalize_attendees(attendees_list: List[str]) -> List[str]:
    """Normalize email addresses for comparison"""
    return sorted([email.lower().strip() for email in attendees_list if email])


def calculate_array_f1(pred_list: List, true_list: List) -> Dict:
    """Calculate precision, recall, F1 for arrays (e.g., attendees)"""
    pred_set = set(normalize_attendees(pred_list)) if pred_list else set()
    true_set = set(normalize_attendees(true_list)) if true_list else set()

    if len(true_set) == 0:
        return {
            "precision": 1.0 if len(pred_set) == 0 else 0.0,
            "recall": 1.0,
            "f1": 1.0 if len(pred_set) == 0 else 0.0
        }

    if len(pred_set) == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    tp = len(pred_set & true_set)
    precision = tp / len(pred_set)
    recall = tp / len(true_set)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {"precision": precision, "recall": recall, "f1": f1}


def normalize_time_confidence(value):
    """Normalize time_confidence values for comparison"""
    # Ground truth uses null for non-meetings
    # Model should output "none" for non-meetings
    # Treat both as equivalent
    if value is None or value == "none":
        return "none"
    return value


def run_inference(model, tokenizer, email: str) -> str:
    """Run model inference on an email"""
    system_prompt = """You are a specialized meeting information extraction system.

TASK: Analyze emails and extract structured meeting details with high precision.

OUTPUT FORMAT (JSON only, no explanations):
{
  "is_meeting": boolean,
  "title": string or null,
  "attendees": array of email addresses,
  "start_time": ISO 8601 string or null,
  "end_time": ISO 8601 string or null,
  "location": string or null,
  "time_confidence": "high" | "medium" | "low" | "none"
}

CRITICAL: ALWAYS return valid JSON. NEVER add explanations."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": email}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id  # ← Add this
        )

    # ========== THE FIX IS HERE ==========
    # Only decode the NEW tokens (not the input prompt)
    generated_tokens = outputs[0][len(inputs['input_ids'][0]):]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    # =====================================

    return response.strip()



In [ ]:
# CELL 2: Run Evaluation Function
# ============================================================================
def evaluate_model(model, tokenizer, test_data, model_name):
    """Evaluate a model on test data"""
    results = []

    for example in tqdm(test_data, desc=f"Evaluating {model_name}"):
        # Extract email and ground truth from messages
        email = [m['content'] for m in example['messages'] if m['role'] == 'user'][0]
        gt_text = [m['content'] for m in example['messages'] if m['role'] == 'assistant'][0]
        ground_truth, _ = parse_json(gt_text)

        # Run inference
        pred_text = run_inference(model, tokenizer, email)
        prediction, pred_valid = parse_json(pred_text)

        results.append({
            'ground_truth': ground_truth,
            'prediction': prediction,
            'pred_valid': pred_valid,
            'pred_text': pred_text,  # ← ADD THIS LINE!!
            'email': email[:200]  # Store snippet for debugging
        })

    return results

## METRICS

In [ ]:
# CELL 3: Calculate Comprehensive Metrics
# ============================================================================
def calculate_comprehensive_metrics(results):
    """Calculate detailed metrics separating meeting vs non-meeting emails"""
    n = len(results)

    # Overall counters
    json_valid = 0
    full_exact_match = 0

    # Meeting detection (binary classification)
    true_positives = 0   # Correctly identified meetings
    false_positives = 0  # Non-meetings labeled as meetings
    true_negatives = 0   # Correctly identified non-meetings
    false_negatives = 0  # Meetings labeled as non-meetings

    # Conditional metrics (only for actual meetings)
    meeting_title_correct = 0
    meeting_location_correct = 0
    meeting_start_time_correct = 0
    meeting_end_time_correct = 0
    meeting_time_confidence_correct = 0
    meeting_attendees_metrics = []

    # Conditional metrics (only for actual non-meetings)
    non_meeting_all_null_correct = 0

    # Track counts
    actual_meetings = 0
    actual_non_meetings = 0

    for r in results:
        pred = r['prediction']
        gt = r['ground_truth']

        # JSON validity
        if r['pred_valid']:
            json_valid += 1
        else:
            continue  # Skip invalid predictions for other metrics

        # Determine actual label
        is_actual_meeting = gt.get('is_meeting', False)
        is_predicted_meeting = pred.get('is_meeting', False)

        if is_actual_meeting:
            actual_meetings += 1
        else:
            actual_non_meetings += 1

        # Meeting detection metrics (confusion matrix)
        if is_actual_meeting and is_predicted_meeting:
            true_positives += 1
        elif not is_actual_meeting and is_predicted_meeting:
            false_positives += 1
        elif not is_actual_meeting and not is_predicted_meeting:
            true_negatives += 1
        elif is_actual_meeting and not is_predicted_meeting:
            false_negatives += 1

        # Normalize time_confidence for comparison
        pred_time_conf = normalize_time_confidence(pred.get('time_confidence'))
        gt_time_conf = normalize_time_confidence(gt.get('time_confidence'))

        # Full exact match (all fields)
        attendees_match = set(normalize_attendees(pred.get('attendees', []))) == set(normalize_attendees(gt.get('attendees', [])))

        if all([
            pred.get('is_meeting') == gt.get('is_meeting'),
            pred.get('title') == gt.get('title'),
            attendees_match,
            pred.get('start_time') == gt.get('start_time'),
            pred.get('end_time') == gt.get('end_time'),
            pred.get('location') == gt.get('location'),
            pred_time_conf == gt_time_conf
        ]):
            full_exact_match += 1

        # Conditional metrics for ACTUAL MEETINGS
        if is_actual_meeting:
            if pred.get('title') == gt.get('title'):
                meeting_title_correct += 1

            if pred.get('location') == gt.get('location'):
                meeting_location_correct += 1

            if pred.get('start_time') == gt.get('start_time'):
                meeting_start_time_correct += 1

            if pred.get('end_time') == gt.get('end_time'):
                meeting_end_time_correct += 1

            if pred_time_conf == gt_time_conf:
                meeting_time_confidence_correct += 1

            att_metrics = calculate_array_f1(
                pred.get('attendees', []),
                gt.get('attendees', [])
            )
            meeting_attendees_metrics.append(att_metrics)

        # Conditional metrics for ACTUAL NON-MEETINGS
        if not is_actual_meeting:
            # Check if all fields are correctly null/empty
            pred_title_null = pred.get('title') is None or pred.get('title') == ''
            pred_attendees_empty = pred.get('attendees', []) == []
            pred_start_null = pred.get('start_time') is None
            pred_end_null = pred.get('end_time') is None
            pred_location_null = pred.get('location') is None or pred.get('location') == ''
            pred_conf_none = normalize_time_confidence(pred.get('time_confidence')) == 'none'

            all_null = all([
                pred_title_null,
                pred_attendees_empty,
                pred_start_null,
                pred_end_null,
                pred_location_null,
                pred_conf_none
            ])

            if all_null:
                non_meeting_all_null_correct += 1

    # Calculate meeting detection metrics
    meeting_precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    meeting_recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    meeting_f1 = 2 * meeting_precision * meeting_recall / (meeting_precision + meeting_recall) if (meeting_precision + meeting_recall) > 0 else 0
    meeting_accuracy = (true_positives + true_negatives) / n

    # Calculate conditional metrics for meetings
    meeting_metrics = {}
    if actual_meetings > 0:
        meeting_metrics = {
            "title_accuracy": meeting_title_correct / actual_meetings,
            "location_accuracy": meeting_location_correct / actual_meetings,
            "start_time_accuracy": meeting_start_time_correct / actual_meetings,
            "end_time_accuracy": meeting_end_time_correct / actual_meetings,
            "time_confidence_accuracy": meeting_time_confidence_correct / actual_meetings,
            "attendees_precision": np.mean([m['precision'] for m in meeting_attendees_metrics]) if meeting_attendees_metrics else 0,
            "attendees_recall": np.mean([m['recall'] for m in meeting_attendees_metrics]) if meeting_attendees_metrics else 0,
            "attendees_f1": np.mean([m['f1'] for m in meeting_attendees_metrics]) if meeting_attendees_metrics else 0,
        }

    # Calculate conditional metrics for non-meetings
    non_meeting_metrics = {}
    if actual_non_meetings > 0:
        non_meeting_metrics = {
            "all_fields_null_accuracy": non_meeting_all_null_correct / actual_non_meetings,
        }

    return {
        "overall": {
            "json_validity": json_valid / n,
            "full_exact_match": full_exact_match / n,
        },
        "meeting_detection": {
            "accuracy": meeting_accuracy,
            "precision": meeting_precision,
            "recall": meeting_recall,
            "f1": meeting_f1,
            "true_positives": true_positives,
            "false_positives": false_positives,
            "true_negatives": true_negatives,
            "false_negatives": false_negatives,
        },
        "meeting_extraction": meeting_metrics,
        "non_meeting_verification": non_meeting_metrics,
        "counts": {
            "total": n,
            "actual_meetings": actual_meetings,
            "actual_non_meetings": actual_non_meetings,
        }
    }

## TESTING (SAMPLE RUN)

In [ ]:
print("="*80)
print("PRE-FLIGHT TESTING - COMPREHENSIVE CHECK")
print("="*80)

# Test on 10 diverse examples
test_sample = test_data[:10]

print(f"\nTesting on {len(test_sample)} examples...")
print("This will take ~2-3 minutes\n")

# ============================================================================
# TEST 1: Both models generate valid JSON
# ============================================================================
print("="*80)
print("TEST 1: Checking JSON validity on sample")
print("="*80)

base_valid_count = 0
ft_valid_count = 0

for i, example in enumerate(test_sample):
    email = [m['content'] for m in example['messages'] if m['role'] == 'user'][0]

    # Test base model
    base_output = run_inference(base_model, base_tokenizer, email)
    base_parsed, base_valid = parse_json(base_output)
    if base_valid:
        base_valid_count += 1

    # Test fine-tuned model
    ft_output = run_inference(ft_model, ft_tokenizer, email)
    ft_parsed, ft_valid = parse_json(ft_output)
    if ft_valid:
        ft_valid_count += 1

    print(f"Example {i+1}: Base={'✅' if base_valid else '❌'}  Fine-tuned={'✅' if ft_valid else '❌'}")

print(f"\nBase Model: {base_valid_count}/{len(test_sample)} valid JSON ({base_valid_count/len(test_sample)*100:.0f}%)")
print(f"Fine-tuned Model: {ft_valid_count}/{len(test_sample)} valid JSON ({ft_valid_count/len(test_sample)*100:.0f}%)")

if base_valid_count == 0 or ft_valid_count == 0:
    print("\n❌ STOP! JSON parsing is failing. Do not run full evaluation.")
else:
    print("\n✅ PASS: Both models generating parseable JSON")

# ============================================================================
# TEST 2: Evaluation function works correctly
# ============================================================================
print("\n" + "="*80)
print("TEST 2: Testing evaluation function pipeline")
print("="*80)

print("\nRunning evaluate_model on 10 examples...")
sample_base_results = evaluate_model(base_model, base_tokenizer, test_sample, "Base")
sample_ft_results = evaluate_model(ft_model, ft_tokenizer, test_sample, "Fine-tuned")

print(f"\n✅ Evaluation function ran successfully")
print(f"Base results: {len(sample_base_results)} examples")
print(f"Fine-tuned results: {len(sample_ft_results)} examples")

# Check that pred_text is stored
if 'pred_text' in sample_ft_results[0]:
    print("✅ pred_text is being stored correctly")
else:
    print("❌ WARNING: pred_text not stored!")

# ============================================================================
# TEST 3: Metrics calculation works
# ============================================================================
print("\n" + "="*80)
print("TEST 3: Testing metrics calculation")
print("="*80)

sample_base_metrics = calculate_comprehensive_metrics(sample_base_results)
sample_ft_metrics = calculate_comprehensive_metrics(sample_ft_results)

print("\n📊 Sample Metrics (10 examples):")
print(f"\nBase Model:")
print(f"  JSON Validity: {sample_base_metrics['overall']['json_validity']:.1%}")
print(f"  Meeting Detection Accuracy: {sample_base_metrics['meeting_detection']['accuracy']:.1%}")
print(f"  Meeting Detection F1: {sample_base_metrics['meeting_detection']['f1']:.1%}")

print(f"\nFine-tuned Model:")
print(f"  JSON Validity: {sample_ft_metrics['overall']['json_validity']:.1%}")
print(f"  Meeting Detection Accuracy: {sample_ft_metrics['meeting_detection']['accuracy']:.1%}")
print(f"  Meeting Detection F1: {sample_ft_metrics['meeting_detection']['f1']:.1%}")

if sample_ft_metrics['overall']['json_validity'] == 0:
    print("\n❌ STOP! All metrics are 0. Do not run full evaluation.")
else:
    print("\n✅ PASS: Metrics are calculating correctly")

# ============================================================================
# TEST 4: Detailed inspection of predictions
# ============================================================================
print("\n" + "="*80)
print("TEST 4: Inspecting sample predictions in detail")
print("="*80)

for i in range(min(3, len(sample_ft_results))):
    print(f"\n{'='*60}")
    print(f"Example {i+1}")
    print(f"{'='*60}")

    gt = sample_ft_results[i]['ground_truth']
    pred = sample_ft_results[i]['prediction']

    print(f"\n📧 Email: {sample_ft_results[i]['email'][:150]}...")

    print(f"\n🎯 Ground Truth:")
    print(f"  is_meeting: {gt.get('is_meeting')}")
    print(f"  title: {gt.get('title')}")
    print(f"  attendees: {len(gt.get('attendees', []))} emails")

    print(f"\n🤖 Fine-tuned Prediction:")
    print(f"  is_meeting: {pred.get('is_meeting')}")
    print(f"  title: {pred.get('title')}")
    print(f"  attendees: {len(pred.get('attendees', []))} emails")

    print(f"\n📊 Match:")
    print(f"  is_meeting: {'✅' if pred.get('is_meeting') == gt.get('is_meeting') else '❌'}")
    print(f"  title: {'✅' if pred.get('title') == gt.get('title') else '❌'}")

# ============================================================================
# TEST 5: Check confusion matrix makes sense
# ============================================================================
print("\n" + "="*80)
print("TEST 5: Confusion Matrix Sanity Check")
print("="*80)

cm = sample_ft_metrics['meeting_detection']
print(f"\nTrue Positives:  {cm['true_positives']}")
print(f"False Positives: {cm['false_positives']}")
print(f"True Negatives:  {cm['true_negatives']}")
print(f"False Negatives: {cm['false_negatives']}")

total = cm['true_positives'] + cm['false_positives'] + cm['true_negatives'] + cm['false_negatives']
print(f"\nTotal: {total} (should be {len(test_sample)})")

if total != len(test_sample):
    print(f"❌ WARNING: Confusion matrix doesn't add up!")
else:
    print("✅ PASS: Confusion matrix totals correct")

# ============================================================================
# FINAL VERDICT
# ============================================================================
print("\n" + "="*80)
print("FINAL VERDICT")
print("="*80)

all_checks_pass = True

# Check 1: JSON validity
if base_valid_count < len(test_sample) * 0.8 or ft_valid_count < len(test_sample) * 0.8:
    print("❌ JSON validity too low (should be >80%)")
    all_checks_pass = False
else:
    print("✅ JSON validity acceptable")

# Check 2: Metrics not all zero
if sample_ft_metrics['overall']['json_validity'] == 0:
    print("❌ All metrics are zero")
    all_checks_pass = False
else:
    print("✅ Metrics are non-zero")

# Check 3: Confusion matrix adds up
if total != len(test_sample):
    print("❌ Confusion matrix error")
    all_checks_pass = False
else:
    print("✅ Confusion matrix correct")

print("\n" + "="*80)
if all_checks_pass:
    print("🎉 ALL TESTS PASSED!")
    print("="*80)
    print("\n👉 SAFE TO RUN FULL EVALUATION")
    print(f"   Estimated time: ~30-60 minutes for {len(test_data)} examples")
    print(f"   Command: evaluate_model(ft_model, ft_tokenizer, test_data, 'Fine-tuned')")
else:
    print("⚠️  ISSUES DETECTED - DO NOT RUN FULL EVALUATION YET")
    print("="*80)
    print("\nFix the issues above before proceeding.")


PRE-FLIGHT TESTING - COMPREHENSIVE CHECK

Testing on 10 examples...
This will take ~2-3 minutes

TEST 1: Checking JSON validity on sample
Example 1: Base=✅  Fine-tuned=✅
Example 2: Base=✅  Fine-tuned=❌
Example 3: Base=✅  Fine-tuned=✅
Example 4: Base=✅  Fine-tuned=✅
Example 5: Base=✅  Fine-tuned=✅
Example 6: Base=✅  Fine-tuned=✅
Example 7: Base=✅  Fine-tuned=✅
Example 8: Base=✅  Fine-tuned=✅
Example 9: Base=✅  Fine-tuned=✅
Example 10: Base=✅  Fine-tuned=✅

Base Model: 10/10 valid JSON (100%)
Fine-tuned Model: 9/10 valid JSON (90%)

✅ PASS: Both models generating parseable JSON

TEST 2: Testing evaluation function pipeline

Running evaluate_model on 10 examples...


Evaluating Fine-tuned: 100%|██████████| 10/10 [01:51<00:00, 11.17s/it]


✅ Evaluation function ran successfully
Base results: 10 examples
Fine-tuned results: 10 examples
✅ pred_text is being stored correctly

TEST 3: Testing metrics calculation

📊 Sample Metrics (10 examples):

Base Model:
  JSON Validity: 100.0%
  Meeting Detection Accuracy: 70.0%
  Meeting Detection F1: 66.7%

Fine-tuned Model:
  JSON Validity: 90.0%
  Meeting Detection Accuracy: 70.0%
  Meeting Detection F1: 75.0%

✅ PASS: Metrics are calculating correctly

TEST 4: Inspecting sample predictions in detail

Example 1

📧 Email: Subject: ferc meeting on northeast rto

----- Forwarded by Sarah Novosel/Corp/Enron on 07/11/2001 04:54 PM -----

	Sarah Novosel
	07/11/2001 04:24 PM
...

🎯 Ground Truth:
  is_meeting: True
  title: FERC Meeting on Northeast RTO
  attendees: 41 emails

🤖 Fine-tuned Prediction:
  is_meeting: False
  title: None
  attendees: 0 emails

📊 Match:
  is_meeting: ❌
  title: ❌

Example 2

📧 Email: Subject: cao staff meetnig

Will he be having one the first tuesday of January

In [ ]:
print("="*80)
print("DEBUGGING EXAMPLE 2 - The Failed Case")
print("="*80)

# Get Example 2
example_2 = test_data[1]
email = [m['content'] for m in example_2['messages'] if m['role'] == 'user'][0]

print("📧 Email:")
print(email[:500])
print("...")

# Run inference and see raw output
print("\n" + "="*80)
print("🤖 Fine-tuned Model Output:")
print("="*80)

raw_output = run_inference(ft_model, ft_tokenizer, email)
print(f"Raw output ({len(raw_output)} chars):")
print(raw_output)

# Try parsing
parsed, valid = parse_json(raw_output)
print(f"\n✅ Valid JSON: {valid}")
print(f"Parsed result:")
print(json.dumps(parsed, indent=2))

# Check if it's actually in the raw output
print("\n" + "="*80)
print("ANALYSIS:")
print("="*80)

if '{' in raw_output and '}' in raw_output:
    print("✓ JSON brackets found in output")
else:
    print("✗ No JSON brackets in output")

if len(raw_output) < 50:
    print("⚠️  Output is very short - model may not have generated properly")

if "assistant" in raw_output.lower():
    print("⚠️  Output still contains chat markers")

DEBUGGING EXAMPLE 2 - The Failed Case
📧 Email:
Subject: cao staff meetnig

Will he be having one the first tuesday of January, 1/2/01?

Sharron Westbrook @ ENRON

12/13/2000 11:14 AM
To: Kent Castleman/NA/Enron@Enron, Sally Beck/HOU/ECT@ECT, Melissa 
Becker@ENRON_DEVELOPMENT, Howard Selzer/Corp/Enron@ENRON, Bob 
Butts@ENRON_DEVELOPMENT, Wes Colwell/HOU/ECT@ECT, Wanda Curry/HOU/EES@EES, 
Fernley Dyson/LON/ECT@ECT, Rodney Faldyn/Corp/Enron@Enron, Rod 
Hayslett@ENRON_DEVELOPMENT, Robert Hermann/Corp/Enron@ENRON, Tod A 
Lindholm/NA/Enron@Enron, 
...

🤖 Fine-tuned Model Output:
Raw output (1409 chars):
{
  "is_meeting": true,
  "title": "CAO Staff meetnig",
  "attendees": [
    "Kent Castleman/NA/Enron@Enron",
    "Sally Beck/HOU/ECT@ECT",
    "Melissa Becker@ENRON_DEVELOPMENT",
    "Howard Selzer/Corp/Enron@ENRON",
    "Bob Butts@ENRON_DEVELOPMENT",
    "Wes Colwell/HOU/ECT@ECT",
    "Wanda Curry/HOU/EES@EES",
    "Fernley Dyson/LON/ECT@ECT",
    "Rodney Faldyn/Corp/Enron@Enron",
    "Rod 

## RUN EVALUATION

In [ ]:
# CELL 4: Run Evaluation on Both Models
# ============================================================================
print("\n" + "="*80)
print("EVALUATING BASE MODEL")
print("="*80)
base_results = evaluate_model(base_model, base_tokenizer, test_data, "Base Model")

print("\n" + "="*80)
print("EVALUATING FINE-TUNED MODEL")
print("="*80)
ft_results = evaluate_model(ft_model, ft_tokenizer, test_data, "Fine-tuned Model")

print("\n Evaluation complete!")



EVALUATING BASE MODEL


Evaluating Base Model: 100%|██████████| 709/709 [55:51<00:00,  4.73s/it]



EVALUATING FINE-TUNED MODEL


Evaluating Fine-tuned Model: 100%|██████████| 709/709 [1:19:24<00:00,  6.72s/it]


 Evaluation complete!


In [ ]:
# CELL 5: Calculate Metrics
# ============================================================================
print("\nCalculating metrics...")
base_metrics = calculate_comprehensive_metrics(base_results)
ft_metrics = calculate_comprehensive_metrics(ft_results)
print("✓ Metrics calculated")



Calculating metrics...
✓ Metrics calculated


In [ ]:
# CELL 6: Display Results - Overall Metrics
# ============================================================================
print("\n" + "="*80)
print("OVERALL METRICS")
print("="*80)

overall_rows = []
for metric in base_metrics['overall'].keys():
    base_val = base_metrics['overall'][metric]
    ft_val = ft_metrics['overall'][metric]
    improvement = ft_val - base_val
    overall_rows.append({
        'Metric': metric,
        'Base': f"{base_val:.2%}",
        'Fine-tuned': f"{ft_val:.2%}",
        'Improvement': f"{improvement:+.2%}"
    })

print(pd.DataFrame(overall_rows).to_string(index=False))



OVERALL METRICS
          Metric    Base Fine-tuned Improvement
   json_validity 100.00%    100.00%      +0.00%
full_exact_match  62.91%     74.75%     +11.85%


In [ ]:
# CELL 7: Display Results - Meeting Detection
# ============================================================================
print("\n" + "="*80)
print("MEETING DETECTION (Binary Classification)")
print("="*80)

detection_rows = []
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    base_val = base_metrics['meeting_detection'][metric]
    ft_val = ft_metrics['meeting_detection'][metric]
    improvement = ft_val - base_val
    detection_rows.append({
        'Metric': metric,
        'Base': f"{base_val:.2%}",
        'Fine-tuned': f"{ft_val:.2%}",
        'Improvement': f"{improvement:+.2%}"
    })

print(pd.DataFrame(detection_rows).to_string(index=False))

# Confusion matrix for fine-tuned model
print("\n Confusion Matrix (Fine-tuned):")
print(f"   True Positives:  {ft_metrics['meeting_detection']['true_positives']:4d}  (Correctly identified meetings)")
print(f"   False Positives: {ft_metrics['meeting_detection']['false_positives']:4d}  (Non-meetings labeled as meetings)")
print(f"   True Negatives:  {ft_metrics['meeting_detection']['true_negatives']:4d}  (Correctly identified non-meetings)")
print(f"   False Negatives: {ft_metrics['meeting_detection']['false_negatives']:4d}  (Meetings missed)")




MEETING DETECTION (Binary Classification)
   Metric   Base Fine-tuned Improvement
 accuracy 90.27%     92.81%      +2.54%
precision 81.53%     84.35%      +2.82%
   recall 86.60%     92.82%      +6.22%
       f1 83.99%     88.38%      +4.39%

 Confusion Matrix (Fine-tuned):
   True Positives:   194  (Correctly identified meetings)
   False Positives:   36  (Non-meetings labeled as meetings)
   True Negatives:   464  (Correctly identified non-meetings)
   False Negatives:   15  (Meetings missed)


In [ ]:
# CELL 8: Display Results - Meeting Extraction (Conditional)
# ============================================================================
print("\n" + "="*80)
print("MEETING EXTRACTION METRICS (Only for Actual Meeting Emails)")
print("="*80)
print(f"Number of meeting emails: {ft_metrics['counts']['actual_meetings']}")
print()

if ft_metrics['meeting_extraction']:
    extraction_rows = []
    for metric in base_metrics['meeting_extraction'].keys():
        base_val = base_metrics['meeting_extraction'][metric]
        ft_val = ft_metrics['meeting_extraction'][metric]
        improvement = ft_val - base_val
        extraction_rows.append({
            'Metric': metric,
            'Base': f"{base_val:.2%}",
            'Fine-tuned': f"{ft_val:.2%}",
            'Improvement': f"{improvement:+.2%}"
        })

    print(pd.DataFrame(extraction_rows).to_string(index=False))
else:
    print("No meeting emails found in test set")




MEETING EXTRACTION METRICS (Only for Actual Meeting Emails)
Number of meeting emails: 209

                  Metric   Base Fine-tuned Improvement
          title_accuracy 23.92%     63.64%     +39.71%
       location_accuracy 75.60%     77.99%      +2.39%
     start_time_accuracy 28.71%     75.60%     +46.89%
       end_time_accuracy 60.29%     88.04%     +27.75%
time_confidence_accuracy 37.32%     72.25%     +34.93%
     attendees_precision 63.83%     81.83%     +18.00%
        attendees_recall 78.64%     82.73%      +4.09%
            attendees_f1 63.74%     81.50%     +17.76%


In [ ]:
# CELL 9: Display Results - Non-Meeting Verification
# ============================================================================
print("\n" + "="*80)
print("NON-MEETING VERIFICATION (Only for Actual Non-Meeting Emails)")
print("="*80)
print(f"Number of non-meeting emails: {ft_metrics['counts']['actual_non_meetings']}")
print()

if ft_metrics['non_meeting_verification']:
    non_meeting_rows = []
    for metric in base_metrics['non_meeting_verification'].keys():
        base_val = base_metrics['non_meeting_verification'][metric]
        ft_val = ft_metrics['non_meeting_verification'][metric]
        improvement = ft_val - base_val
        non_meeting_rows.append({
            'Metric': metric,
            'Base': f"{base_val:.2%}",
            'Fine-tuned': f"{ft_val:.2%}",
            'Improvement': f"{improvement:+.2%}"
        })

    print(pd.DataFrame(non_meeting_rows).to_string(index=False))
else:
    print("No non-meeting emails found in test set")



NON-MEETING VERIFICATION (Only for Actual Non-Meeting Emails)
Number of non-meeting emails: 500

                  Metric   Base Fine-tuned Improvement
all_fields_null_accuracy 88.40%     92.80%      +4.40%


In [ ]:
# CELL 10: Summary & Save Results
# ============================================================================
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print(f"\n Dataset Composition:")
print(f"  Total emails: {ft_metrics['counts']['total']}")
print(f"  Meeting emails: {ft_metrics['counts']['actual_meetings']} ({ft_metrics['counts']['actual_meetings']/ft_metrics['counts']['total']*100:.1f}%)")
print(f"  Non-meeting emails: {ft_metrics['counts']['actual_non_meetings']} ({ft_metrics['counts']['actual_non_meetings']/ft_metrics['counts']['total']*100:.1f}%)")

print(f"\n Key Improvements (Fine-tuned vs Base):")
print(f"  JSON Validity: {base_metrics['overall']['json_validity']:.1%} → {ft_metrics['overall']['json_validity']:.1%} ({ft_metrics['overall']['json_validity']-base_metrics['overall']['json_validity']:+.1%})")
print(f"  Meeting Detection F1: {base_metrics['meeting_detection']['f1']:.1%} → {ft_metrics['meeting_detection']['f1']:.1%} ({ft_metrics['meeting_detection']['f1']-base_metrics['meeting_detection']['f1']:+.1%})")
print(f"  Full Exact Match: {base_metrics['overall']['full_exact_match']:.1%} → {ft_metrics['overall']['full_exact_match']:.1%} ({ft_metrics['overall']['full_exact_match']-base_metrics['overall']['full_exact_match']:+.1%})")

if ft_metrics['meeting_extraction']:
    print(f"  Attendees F1 (meetings only): {base_metrics['meeting_extraction']['attendees_f1']:.1%} → {ft_metrics['meeting_extraction']['attendees_f1']:.1%} ({ft_metrics['meeting_extraction']['attendees_f1']-base_metrics['meeting_extraction']['attendees_f1']:+.1%})")

# Save results
output_dir = f"{CHECKPOINT_DIR}/evaluation_results"
os.makedirs(output_dir, exist_ok=True)

with open(f"{output_dir}/base_metrics_detailed.json", 'w') as f:
    # Convert numpy types to native Python types for JSON serialization
    def convert_to_native(obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {k: convert_to_native(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_to_native(item) for item in obj]
        return obj

    json.dump(convert_to_native(base_metrics), f, indent=2)

with open(f"{output_dir}/finetuned_metrics_detailed.json", 'w') as f:
    json.dump(convert_to_native(ft_metrics), f, indent=2)

print(f"\n Results saved to: {output_dir}")
print(f"  - base_metrics_detailed.json")
print(f"  - finetuned_metrics_detailed.json")

print("\n" + "="*80)
print(" EVALUATION COMPLETE!")
print("="*80)


SUMMARY

 Dataset Composition:
  Total emails: 709
  Meeting emails: 209 (29.5%)
  Non-meeting emails: 500 (70.5%)

 Key Improvements (Fine-tuned vs Base):
  JSON Validity: 100.0% → 100.0% (+0.0%)
  Meeting Detection F1: 84.0% → 88.4% (+4.4%)
  Full Exact Match: 62.9% → 74.8% (+11.8%)
  Attendees F1 (meetings only): 63.7% → 81.5% (+17.8%)

 Results saved to: /content/drive/MyDrive/qwen-meeting-extraction/evaluation_results
  - base_metrics_detailed.json
  - finetuned_metrics_detailed.json

 EVALUATION COMPLETE!


## ADVANCED METRICS

In [ ]:
# Install additional packages for enhanced metrics
!pip install -q evaluate bert-score sentence-transformers rouge_score nltk

  Preparing metadata (setup.py) ... done


In [ ]:
# Import enhanced metrics libraries with better error handling
import evaluate
from bert_score import score as bert_score_fn
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

print(" Enhanced metrics libraries loaded")

# Initialize models (cached after first use)
print("Loading SBERT model (one-time download)...")
_SBERT_MODEL = SentenceTransformer("all-MiniLM-L6-v2")
print("SBERT model loaded")

print("Loading ROUGE metric...")
rouge_metric = evaluate.load("rouge")
print(" ROUGE metric loaded")

print("\n All enhanced metrics ready!")

 Enhanced metrics libraries loaded
Loading SBERT model (one-time download)...
SBERT model loaded
Loading ROUGE metric...


 ROUGE metric loaded

 All enhanced metrics ready!


In [ ]:
# =============================================================================
# ENHANCED METRIC FUNCTIONS
# =============================================================================

def metric_rouge(preds, refs):
    """ROUGE metric on full JSON strings"""
    try:
        pred_strs = [json.dumps(p, sort_keys=True) for p in preds]
        ref_strs = [json.dumps(r, sort_keys=True) for r in refs]
        return rouge_metric.compute(predictions=pred_strs, references=ref_strs)
    except Exception as e:
        print(f" ROUGE failed: {e}")
        return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}


def metric_bertscore(preds, refs):
    """BERTScore on meeting titles"""
    try:
        pred_titles = [str(p.get("title") or "") for p in preds]
        ref_titles = [str(r.get("title") or "") for r in refs]
        P, R, F1 = bert_score_fn(pred_titles, ref_titles, lang="en", verbose=False)
        return {
            "bertscore_precision": float(P.mean()),
            "bertscore_recall": float(R.mean()),
            "bertscore_f1": float(F1.mean()),
        }
    except Exception as e:
        print(f"BERTScore failed: {e}")
        return {"bertscore_precision": 0.0, "bertscore_recall": 0.0, "bertscore_f1": 0.0}


def metric_sbert_similarity(preds, refs):
    """SBERT cosine similarity on meeting titles"""
    try:
        pred_titles = [str(p.get("title") or "") for p in preds]
        ref_titles = [str(r.get("title") or "") for r in refs]

        pred_embs = _SBERT_MODEL.encode(pred_titles, convert_to_numpy=True)
        ref_embs = _SBERT_MODEL.encode(ref_titles, convert_to_numpy=True)

        scores = [
            float(cos_sim(p.reshape(1, -1), r.reshape(1, -1))[0][0])
            for p, r in zip(pred_embs, ref_embs)
        ]
        return {
            "sbert_cosine_mean": float(np.mean(scores)),
            "sbert_cosine_min": float(np.min(scores)),
        }
    except Exception as e:
        print(f" SBERT failed: {e}")
        return {"sbert_cosine_mean": 0.0, "sbert_cosine_min": 0.0}


def metric_field_exact_match(preds, refs):
    """Field-level exact match for each field"""
    fields = ["title", "location", "start_time", "end_time", "time_confidence"]
    results = {}

    for field in fields:
        correct = sum(
            1 for p, r in zip(preds, refs)
            if str(p.get(field) or "").strip().lower() ==
               str(r.get(field) or "").strip().lower()
        )
        results[f"exact_match_{field}"] = correct / len(preds) if preds else 0.0

    # All fields correct
    all_correct = sum(
        1 for p, r in zip(preds, refs)
        if all(
            str(p.get(f) or "").strip().lower() ==
            str(r.get(f) or "").strip().lower()
            for f in fields
        ) and set(normalize_attendees(p.get('attendees', []))) ==
             set(normalize_attendees(r.get('attendees', [])))
    )
    results["exact_match_all_fields"] = all_correct / len(preds) if preds else 0.0

    return results


def metric_time_accuracy(preds, refs):
    """Time extraction accuracy"""
    start_correct = sum(
        1 for p, r in zip(preds, refs)
        if p.get("start_time") == r.get("start_time")
    )
    end_correct = sum(
        1 for p, r in zip(preds, refs)
        if p.get("end_time") == r.get("end_time")
    )
    both_correct = sum(
        1 for p, r in zip(preds, refs)
        if p.get("start_time") == r.get("start_time") and
           p.get("end_time") == r.get("end_time")
    )

    return {
        "start_time_accuracy": start_correct / len(preds) if preds else 0.0,
        "end_time_accuracy": end_correct / len(preds) if preds else 0.0,
        "both_times_accuracy": both_correct / len(preds) if preds else 0.0,
    }


def _token_f1(pred_str, ref_str):
    """Calculate token-level F1"""
    def tokenise(s):
        return re.findall(r"\w+", s.lower())

    pred_tokens = tokenise(str(pred_str or ""))
    ref_tokens = tokenise(str(ref_str or ""))

    if not pred_tokens and not ref_tokens:
        return 1.0, 1.0, 1.0
    if not pred_tokens or not ref_tokens:
        return 0.0, 0.0, 0.0

    pred_counts = {}
    for t in pred_tokens:
        pred_counts[t] = pred_counts.get(t, 0) + 1

    ref_counts = {}
    for t in ref_tokens:
        ref_counts[t] = ref_counts.get(t, 0) + 1

    common = sum(min(pred_counts.get(t, 0), ref_counts.get(t, 0))
                 for t in ref_counts)

    precision = common / len(pred_tokens)
    recall = common / len(ref_tokens)
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)
    return precision, recall, f1


def metric_token_f1(preds, refs):
    """Token-level F1 on meeting titles"""
    precisions, recalls, f1s = [], [], []
    for p, r in zip(preds, refs):
        prec, rec, f1 = _token_f1(p.get("title"), r.get("title"))
        precisions.append(prec)
        recalls.append(rec)
        f1s.append(f1)

    return {
        "token_f1_precision": float(np.mean(precisions)),
        "token_f1_recall": float(np.mean(recalls)),
        "token_f1_f1": float(np.mean(f1s)),
    }


def metric_attendee_f1(preds, refs):
    """Attendee extraction F1"""
    all_precision = []
    all_recall = []
    all_f1 = []

    for p, r in zip(preds, refs):
        metrics = calculate_array_f1(
            p.get('attendees', []),
            r.get('attendees', [])
        )
        all_precision.append(metrics['precision'])
        all_recall.append(metrics['recall'])
        all_f1.append(metrics['f1'])

    return {
        "attendee_precision": float(np.mean(all_precision)),
        "attendee_recall": float(np.mean(all_recall)),
        "attendee_f1": float(np.mean(all_f1)),
    }

print(" Enhanced metric functions defined")

 Enhanced metric functions defined


In [ ]:
# =============================================================================
# ENHANCED METRICS CALCULATOR
# =============================================================================

def add_enhanced_metrics(base_metrics_dict, results):
    """
    Add enhanced metrics to your existing metrics dictionary
    Assumes you already ran calculate_comprehensive_metrics()
    """
    # Collect valid predictions and references
    valid_preds = []
    valid_refs = []

    for r in results:
        if r['pred_valid']:
            valid_preds.append(r['prediction'])
            valid_refs.append(r['ground_truth'])

    if not valid_preds:
        print(" No valid predictions found!")
        return base_metrics_dict

    print(f" Computing enhanced metrics on {len(valid_preds)} valid predictions...")
    enhanced_metrics = {}

    print("   • ROUGE...")
    enhanced_metrics.update(metric_rouge(valid_preds, valid_refs))

    print("   • BERTScore (titles)...")
    enhanced_metrics.update(metric_bertscore(valid_preds, valid_refs))

    print("   • SBERT Cosine Similarity (titles)...")
    enhanced_metrics.update(metric_sbert_similarity(valid_preds, valid_refs))

    print("   • Field-Level Exact Match...")
    enhanced_metrics.update(metric_field_exact_match(valid_preds, valid_refs))

    print("   • Time Accuracy...")
    enhanced_metrics.update(metric_time_accuracy(valid_preds, valid_refs))

    print("   • Token F1 (titles)...")
    enhanced_metrics.update(metric_token_f1(valid_preds, valid_refs))

    print("   • Attendee F1...")
    enhanced_metrics.update(metric_attendee_f1(valid_preds, valid_refs))

    # Add to existing metrics
    base_metrics_dict["enhanced"] = enhanced_metrics

    print(" Enhanced metrics added!")
    return base_metrics_dict

print(" Enhanced metrics calculator ready")

 Enhanced metrics calculator ready


In [ ]:
# =============================================================================
# RUN ENHANCED METRICS
# =============================================================================
# This assumes you already have:
# - base_results (from evaluate_model on base model)
# - ft_results (from evaluate_model on fine-tuned model)
# - base_metrics (from calculate_comprehensive_metrics)
# - ft_metrics (from calculate_comprehensive_metrics)

print("="*80)
print("ADDING ENHANCED METRICS TO EXISTING RESULTS")
print("="*80)

# Add enhanced metrics to base model results
print("\nBase Model:")
base_metrics_enhanced = add_enhanced_metrics(base_metrics.copy(), base_results)

# Add enhanced metrics to fine-tuned model results
print("\n Fine-tuned Model:")
ft_metrics_enhanced = add_enhanced_metrics(ft_metrics.copy(), ft_results)

print("\nEnhanced metrics computed for both models!")

ADDING ENHANCED METRICS TO EXISTING RESULTS

Base Model:
 Computing enhanced metrics on 709 valid predictions...
   • ROUGE...
   • BERTScore (titles)...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   • SBERT Cosine Similarity (titles)...
   • Field-Level Exact Match...
   • Time Accuracy...
   • Token F1 (titles)...
   • Attendee F1...
 Enhanced metrics added!

 Fine-tuned Model:
 Computing enhanced metrics on 709 valid predictions...
   • ROUGE...
   • BERTScore (titles)...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   • SBERT Cosine Similarity (titles)...
   • Field-Level Exact Match...
   • Time Accuracy...
   • Token F1 (titles)...
   • Attendee F1...
 Enhanced metrics added!

Enhanced metrics computed for both models!


In [ ]:
# =============================================================================
# PRINT ENHANCED METRICS COMPARISON
# =============================================================================

def print_enhanced_comparison(base_m, ft_m):
    """Print comparison including enhanced metrics"""

    print("\n" + "="*80)
    print(" ENHANCED METRICS COMPARISON")
    print("="*80)

    if 'enhanced' not in base_m or 'enhanced' not in ft_m:
        print(" Enhanced metrics not found!")
        return

    enhanced = base_m['enhanced']
    ft_enhanced = ft_m['enhanced']

    print(f"\n{'Metric':<40} {'Base':<15} {'Fine-tuned':<15} {'Δ':<10}")
    print("-" * 80)

    # ROUGE metrics
    if 'rouge1' in enhanced:
        print("\n ROUGE (N-gram Overlap)")
        for metric in ['rouge1', 'rouge2', 'rougeL']:
            if metric in enhanced:
                base_val = enhanced[metric]
                ft_val = ft_enhanced[metric]
                print(f"{metric.upper():<40} "
                      f"{base_val:>14.4f} "
                      f"{ft_val:>14.4f} "
                      f"{ft_val - base_val:>+9.4f}")

    # BERTScore
    if 'bertscore_f1' in enhanced:
        print("\n BERTScore (Semantic Similarity on Titles)")
        for metric in ['bertscore_precision', 'bertscore_recall', 'bertscore_f1']:
            base_val = enhanced[metric]
            ft_val = ft_enhanced[metric]
            name = metric.replace('bertscore_', '').title()
            print(f"  {name:<38} "
                  f"{base_val:>14.4f} "
                  f"{ft_val:>14.4f} "
                  f"{ft_val - base_val:>+9.4f}")

    # SBERT
    if 'sbert_cosine_mean' in enhanced:
        print("\n SBERT Cosine Similarity (Dense Embeddings on Titles)")
        for metric in ['sbert_cosine_mean', 'sbert_cosine_min']:
            base_val = enhanced[metric]
            ft_val = ft_enhanced[metric]
            name = metric.replace('sbert_cosine_', '').title()
            print(f"  {name:<38} "
                  f"{base_val:>14.4f} "
                  f"{ft_val:>14.4f} "
                  f"{ft_val - base_val:>+9.4f}")

    # Field-level exact match
    if 'exact_match_title' in enhanced:
        print("\n Field-Level Exact Match (FactScore-style)")
        for field in ['title', 'location', 'start_time', 'end_time', 'time_confidence']:
            metric = f'exact_match_{field}'
            if metric in enhanced:
                base_val = enhanced[metric]
                ft_val = ft_enhanced[metric]
                print(f"  {field.replace('_', ' ').title():<38} "
                      f"{base_val:>14.1%} "
                      f"{ft_val:>14.1%} "
                      f"{ft_val - base_val:>+9.1%}")

        if 'exact_match_all_fields' in enhanced:
            base_val = enhanced['exact_match_all_fields']
            ft_val = ft_enhanced['exact_match_all_fields']
            print(f"  {'All Fields Correct':<38} "
                  f"{base_val:>14.1%} "
                  f"{ft_val:>14.1%} "
                  f"{ft_val - base_val:>+9.1%}")

    # Time accuracy
    if 'start_time_accuracy' in enhanced:
        print("\n Time Extraction Accuracy")
        for metric in ['start_time_accuracy', 'end_time_accuracy', 'both_times_accuracy']:
            base_val = enhanced[metric]
            ft_val = ft_enhanced[metric]
            name = metric.replace('_accuracy', '').replace('_', ' ').title()
            print(f"  {name:<38} "
                  f"{base_val:>14.1%} "
                  f"{ft_val:>14.1%} "
                  f"{ft_val - base_val:>+9.1%}")

    # Token F1
    if 'token_f1_f1' in enhanced:
        print("\n Token F1 (SQuAD-style, on Titles)")
        for metric in ['token_f1_precision', 'token_f1_recall', 'token_f1_f1']:
            base_val = enhanced[metric]
            ft_val = ft_enhanced[metric]
            name = metric.replace('token_f1_', '').title()
            print(f"  {name:<38} "
                  f"{base_val:>14.4f} "
                  f"{ft_val:>14.4f} "
                  f"{ft_val - base_val:>+9.4f}")

    # Attendee F1
    if 'attendee_f1' in enhanced:
        print("\n Attendee Extraction F1")
        for metric in ['attendee_precision', 'attendee_recall', 'attendee_f1']:
            base_val = enhanced[metric]
            ft_val = ft_enhanced[metric]
            name = metric.replace('attendee_', '').title()
            print(f"  {name:<38} "
                  f"{base_val:>14.4f} "
                  f"{ft_val:>14.4f} "
                  f"{ft_val - base_val:>+9.4f}")

    print("\n" + "="*80)

# Print the comparison
print_enhanced_comparison(base_metrics_enhanced, ft_metrics_enhanced)


 ENHANCED METRICS COMPARISON

Metric                                   Base            Fine-tuned      Δ         
--------------------------------------------------------------------------------

 ROUGE (N-gram Overlap)
ROUGE1                                           0.8567         0.9475   +0.0909
ROUGE2                                           0.7579         0.9197   +0.1617
ROUGEL                                           0.8524         0.9456   +0.0933

 BERTScore (Semantic Similarity on Titles)
  Precision                                      0.2210         0.2679   +0.0469
  Recall                                         0.2158         0.2659   +0.0502
  F1                                             0.2182         0.2668   +0.0486

 SBERT Cosine Similarity (Dense Embeddings on Titles)
  Mean                                           0.8462         0.9224   +0.0762
  Min                                           -0.0217        -0.0217   +0.0000

 Field-Level Exact Match (FactS

## Note that the null null comparison is yeilding low bert score above , with meeting only emails , the bert score is below

In [ ]:
# =============================================================================
# SAVE ENHANCED RESULTS
# =============================================================================

import pickle

# Save enhanced metrics
results_dir = "/content/drive/MyDrive/meeting-extraction-results"
os.makedirs(results_dir, exist_ok=True)

# Save to JSON
with open(f"{results_dir}/base_metrics_enhanced.json", 'w') as f:
    json.dump(base_metrics_enhanced, f, indent=2)

with open(f"{results_dir}/ft_metrics_enhanced.json", 'w') as f:
    json.dump(ft_metrics_enhanced, f, indent=2)

print(f" Enhanced metrics saved to {results_dir}")

 Enhanced metrics saved to /content/drive/MyDrive/meeting-extraction-results


In [ ]:
c# =============================================================================
# CORRECT BERTSCORE - Same Examples for Both Models
# =============================================================================

print("="*80)
print("BERTSCORE COMPARISON - Same Examples (Meetings Only)")
print("="*80)

# Collect predictions from BOTH models for the SAME examples
paired_results = []

for base_r, ft_r in zip(base_results, ft_results):
    # Only include if:
    # 1. Both predictions are valid JSON
    # 2. Ground truth is a meeting
    if (base_r['pred_valid'] and
        ft_r['pred_valid'] and
        base_r['ground_truth'].get('is_meeting') == True):

        paired_results.append({
            'base_pred': base_r['prediction'],
            'ft_pred': ft_r['prediction'],
            'ground_truth': base_r['ground_truth']
        })

print(f"\nFound {len(paired_results)} meetings with valid predictions from BOTH models")

if paired_results:
    # Base model titles
    base_pred_titles = [str(r['base_pred'].get("title") or "") for r in paired_results]

    # Fine-tuned model titles
    ft_pred_titles = [str(r['ft_pred'].get("title") or "") for r in paired_results]

    # Ground truth titles (same for both)
    ref_titles = [str(r['ground_truth'].get("title") or "") for r in paired_results]

    # Compute BERTScore for BOTH models on SAME examples
    print("\nComputing BERTScore for base model...")
    base_P, base_R, base_F1 = bert_score_fn(base_pred_titles, ref_titles,
                                             lang="en", verbose=False)

    print("Computing BERTScore for fine-tuned model...")
    ft_P, ft_R, ft_F1 = bert_score_fn(ft_pred_titles, ref_titles,
                                       lang="en", verbose=False)

    # Results
    base_bertscore = float(base_F1.mean())
    ft_bertscore = float(ft_F1.mean())

    print("\n" + "="*80)
    print(f"RESULTS - BERTScore F1 (Same {len(paired_results)} Meeting Examples)")
    print("="*80)

    print(f"\nBase Model:        {base_bertscore:.4f}")
    print(f"Fine-tuned Model:  {ft_bertscore:.4f}")
    print(f"Improvement (Δ):   {ft_bertscore - base_bertscore:+.4f}")

    if base_bertscore > 0:
        improvement_pct = ((ft_bertscore - base_bertscore) / base_bertscore) * 100
        print(f"Improvement (%):   {improvement_pct:+.1f}%")

    print("\n" + "="*80)


BERTSCORE COMPARISON - Same Examples (Meetings Only)

Found 209 meetings with valid predictions from BOTH models

Computing BERTScore for base model...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Computing BERTScore for fine-tuned model...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



RESULTS - BERTScore F1 (Same 209 Meeting Examples)

Base Model:        0.7403
Fine-tuned Model:  0.9051
Improvement (Δ):   +0.1648
Improvement (%):   +22.3%

